In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect("../results/results_am_benchmark.db")

In [9]:
pd.read_sql_query(" SELECT * FROM agent_results WHERE model_label NOT IN ('0','1','2','3', '4') AND agent_name NOT LIKE 'uam_ga1%' LIMIT 50", conn)

,id,agent_name,claim_id,benchmark_name,original_label,model_label,is_correct,total_tokens,prompt_tokens,completion_tokens,time_thought,raw_output,created_at,model_name
0,57082,uam_ga3,222,am_benchmark,2,Aby ocenić prawdziwość wypowiedzi dotyczącej k...,0,1537,1012,525,8.014973,Aby ocenić prawdziwość wypowiedzi dotyczącej k...,2026-04-12T08:45:06.346015+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
1,57132,uam_ga3,272,am_benchmark,0,W opisie z Wikipedii nie ma informacji dotyczą...,0,1262,970,292,5.929570,W opisie z Wikipedii nie ma informacji dotyczą...,2026-04-12T08:49:43.763468+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
2,57188,uam_ga3,328,am_benchmark,3,Output: 4,0,968,963,5,0.680901,Output: 4,2026-04-12T08:53:22.888643+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
3,57681,uam_ga3,821,am_benchmark,0,Oceniając pytanie i wykorzystując dostarczony ...,0,1707,1032,675,9.566356,Oceniając pytanie i wykorzystując dostarczony ...,2026-04-12T09:24:16.347356+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
4,57697,uam_ga3,837,am_benchmark,3,Brak wystarczających informacji w kontekście z...,0,617,545,72,1.456416,Brak wystarczających informacji w kontekście z...,2026-04-12T09:25:26.358162+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
5,57750,uam_ga3,890,am_benchmark,0,Z kontekstu z Wikipedii nie wynika bezpośredni...,0,1681,1212,469,7.152367,Z kontekstu z Wikipedii nie wynika bezpośredni...,2026-04-12T09:29:12.583813+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
6,57791,uam_ga3,931,am_benchmark,1,"Niestety, kontekst z Wikipedii nie dostarcza w...",0,1214,874,340,4.972564,"Niestety, kontekst z Wikipedii nie dostarcza w...",2026-04-12T09:31:48.472838+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
7,57951,uam_ga3,1091,am_benchmark,1,Ocena pytania i odpowiedzi nie jest możliwa na...,0,942,785,157,3.043388,Ocena pytania i odpowiedzi nie jest możliwa na...,2026-04-12T09:39:27.639667+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
8,58006,uam_ga3,1146,am_benchmark,1,Na podstawie dostarczonego kontekstu z Wikiped...,0,1127,707,420,6.543879,Na podstawie dostarczonego kontekstu z Wikiped...,2026-04-12T09:41:41.508695+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...
9,58052,uam_ga3,1192,am_benchmark,3,Kontekst zawiera informacje o mięśniach oddech...,0,1649,1328,321,4.537557,Kontekst zawiera informacje o mięśniach oddech...,2026-04-12T09:43:42.105811+00:00,hf.co/speakleash/Bielik-11B-v2.3-Instruct-GGUF...


## Analysis

Cells below use SQL aggregations before pulling data into pandas, keeping memory usage low.  
`FILTER` excludes rows where the model output couldn't be parsed to a valid label, and the early `uam_ga1` experiments.


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Rows with parseable labels only, excluding early uam_ga1 experiments
FILTER = "model_label IN ('0','1','2','3','4') AND agent_name NOT LIKE 'uam_ga1%'"

# Quick data quality summary
df_quality = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN model_label IN ('0','1','2','3','4') THEN 1 ELSE 0 END) AS clean_rows,
        SUM(CASE WHEN model_label NOT IN ('0','1','2','3','4') THEN 1 ELSE 0 END) AS unparseable_rows,
        COUNT(DISTINCT agent_name) AS agents,
        COUNT(DISTINCT model_name) AS models,
        MIN(created_at) AS earliest,
        MAX(created_at) AS latest
    FROM agent_results
""", conn)
df_quality


In [ ]:

df_overview = pd.read_sql_query(f"""
    SELECT
        agent_name,
        model_name,
        COUNT(*) AS n_runs,
        ROUND(AVG(is_correct) * 100, 1) AS accuracy_pct,
        ROUND(AVG(total_tokens)) AS avg_tokens,
        ROUND(AVG(time_thought), 2) AS avg_latency_s,
        ROUND(AVG(prompt_tokens)) AS avg_prompt_tokens,
        ROUND(AVG(completion_tokens)) AS avg_completion_tokens
    FROM agent_results
    WHERE {FILTER}
    GROUP BY agent_name, model_name
    ORDER BY accuracy_pct DESC
""", conn)
df_overview


In [ ]:

df_acc = df_overview.sort_values("accuracy_pct")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_acc["agent_name"], df_acc["accuracy_pct"], color=sns.color_palette("muted"))
ax.axvline(20, color="red", linestyle="--", linewidth=0.9, label="Random baseline (5 classes = 20%)")
ax.set_xlabel("Accuracy (%)")
ax.set_title("Accuracy by Agent")
ax.legend()
for bar, val in zip(bars, df_acc["accuracy_pct"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2, f"{val}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()
